<a href="https://colab.research.google.com/github/cechmankovaanna-hue/PyDeck/blob/main/PyDeck.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Úvod do knihovny PyDeck

**PyDeck** je Python wrapper pro 3D vizualizační framework **deck.gl** (od společnosti Uber). Slouží k interaktivní vizualizaci velikých objemů prostory definovaných dat (geodat) přímo v prostředí Jupyter Notebooku a Google Colab.

### Hlavní přednosti:
* **Vysoký výkon díky WebGL:** Výpočty a vykreslování probíhají přímo na grafické kartě (GPU), což umožňuje plynulé zobrazení i pro miliony datových bodů.
* **3D vizualizace a vrstvení:** Podporuje naklápění kamery, 3D sloupce, hexagonální mřížky hustoty a 3D oblouky (např. trasy a migrace).
* **Snadné propojení s Pandas:** Přímo načítá data z `pandas.DataFrame`.

## Načtení a příprava dat

Pro ukázku využijeme živá data o zemětřeseních za poslední měsíc z **USGS (U.S. Geological Survey)**.
Data obsahují geografické souřadnice, magnitudo (sílu) a hloubku jednotlivých otřesů.

In [ ]:
import pandas as pd

# Načtení dat přímo z živého API (CSV format)
DATA_URL = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_month.csv"
df = pd.read_csv(DATA_URL)

# Odstranění řádků s chybějícími souřadnicemi nebo magnitudem
df = df.dropna(subset=['latitude', 'longitude', 'mag'])

# Náhled prvních 5 řádků
df[['place', 'latitude', 'longitude', 'mag', 'depth']].head()

## Úkol 1: Základní vizualizace bodů (ScatterplotLayer)

Nejjednodušším způsobem vizualizace je zobrazení jednotlivých událostí jako bodů na mapě.
K tomu slouží vrstva `ScatterplotLayer`, kde definujeme souřadnice, poloměr bodu a jeho barvu.

In [ ]:
!pip install pydeck pandas

In [ ]:
import pydeck as pdk

# 1. Definice úvodního pohledu kamery (střed, zoom, naklonění)
view_state = pdk.ViewState(
    latitude=20.0,
    longitude=0.0,
    zoom=1.5
)

# 2. Definice bodové vrstvy
scatterplot_layer = pdk.Layer(
    "ScatterplotLayer",
    data=df,
    get_position=["longitude", "latitude"],
    get_radius="mag * 25000",        # Poloměr bodu závisí na síle zemětřesení
    get_fill_color="[255, (10 - mag) * 20, 0, 180]", # RGBA barva (červeno-oranžová)
    pickable=True                     # Umožní interaktivní klikání/najetí myší
)

# 3. Vykreslení mapy s vysvětlivkou (Tooltip)
r = pdk.Deck(
    layers=[scatterplot_layer],
    initial_view_state=view_state,
    tooltip={"text": "Místo: {place}\nMagnitudo: {mag}"}
)

r.show()
#r.to_html("ukol1_scatterplot.html", notebook_display=True)

## Úkol 2: 3D Sloupcová mapa (ColumnLayer)

Přejdeme od 2D bodů ke 3D vizualizaci. Pomocí `ColumnLayer` zobrazíme každé zemětřesení jako 3D sloupec.
* **Výška sloupu (`get_elevation`):** Úměrná síle zemětřesení (`mag`).
* **Barva sloupu (`get_fill_color`):** Vyjadřuje hloubku ohniska (`depth`).
* **Naklonění kamery (`pitch`):** Nastaveno na 50° pro zřetelný 3D efekt.

In [ ]:
# 1. Definice pohledu s nakloněním kamery (pitch=50)
view_state_3d = pdk.ViewState(
    latitude=20.0,
    longitude=0.0,
    zoom=1.5,
    pitch=50,       # Naklonění kamery ve stupních (0 = kolmý 2D pohled)
)

# 2. Definice 3D sloupcové vrstvy
column_layer = pdk.Layer(
    "ColumnLayer",
    data=df,
    get_position=["longitude", "latitude"],
    get_elevation="mag",             # Výška sloupu podle síly zemětřesení
    elevation_scale=100000,          # Násobič výšky v metrech
    radius=30000,                    # Poloměr sloupu v metrech
    # Dynamická barva (RGBA): Červená složka roste s hloubkou (depth)
    get_fill_color="[depth * 3, 200 - (depth * 2), 255 - (depth * 3), 200]",
    pickable=True,
    extruded=True                    # Klíčový parametr pro zapnutí 3D vytažení
)

# 3. Vykreslení mapy
r2 = pdk.Deck(
    layers=[column_layer],
    initial_view_state=view_state_3d,
    tooltip={"text": "Místo: {place}\nMagnitudo: {mag}\nHloubka: {depth} km"}
)

r2.to_html("ukol2_column.html", notebook_display=True)

## Úkol 3: Agregace dat do 3D Hexagonů (HexagonLayer)

Pokud máme tisíce překrývajících se bodů (např. kolem Tichého oceánu), mapa se stává nepřehlednou.
`HexagonLayer` automaticky shlukuje blízké body do 3D hexagonální mřížky:
* **Výška hexagonu:** Odpovídá počtu zemětřesení v dané buňce.
* **Barva hexagonu:** Vyjadřuje hustotu incidentů (od žluté po červenou).

In [ ]:
# 1. Pohled nakloněný na oblast s vysokou seismickou aktivitou (např. Tichý oceán / Japonsko)
view_state_hex = pdk.ViewState(
    latitude=35.0,
    longitude=135.0,
    zoom=3,
    pitch=55,
    bearing=-10      # Mírné natočení pohledu do strany
)

# 2. Definice hexagonální vrstvy
hexagon_layer = pdk.Layer(
    "HexagonLayer",
    data=df,
    get_position=["longitude", "latitude"],
    radius=100000,                  # Poloměr hexagonu v metrech (100 km)
    elevation_scale=1000,           # Násobič výšky hexagonu podle počtu bodů uvnitř
    elevation_range=[0, 3000],       # Rozsah výšky hexagonů
    extruded=True,                  # Zapnutí 3D vytažení
    pickable=True,
    auto_highlight=True             # Zvýrazní hexagon po najetí myší
)

# 3. Vykreslení mapy
r3 = pdk.Deck(
    layers=[hexagon_layer],
    initial_view_state=view_state_hex,
    tooltip={"text": "Počet zemětřesení v oblasti: {count}"}
)

r3.to_html("ukol3_hexagon.html", notebook_display=True)

## Úkol 4: Propojení světových leteckých uzlů (ArcLayer)

V této ukázce vizualizujeme vzájemné propojení mezi 6 hlavními světovými letišti (New York, Londýn, Tokio, Dubaj, Sydney a Kapské Město).
* **Vzájemné linky:** Zobrazují se pouze přímé trasy spojující tyto uzly.
* **Barevné rozlišení:** Každé odletové letiště má svou unikátní barvu.
* **3D výška oblouku:** PyDeck výšku paraboly spočítá automaticky podle vzdálenosti mezi městy.

In [ ]:
import pandas as pd
import pydeck as pdk

# 1. Definujeme 6 hlavních letišť se souřadnicemi a unikátní barvou
airports_list = [
    {'name': 'New York (JFK)', 'lat': 40.6413, 'lon': -73.7781, 'color': [255, 0, 128, 200]},     # Růžová
    {'name': 'London (LHR)',   'lat': 51.4700, 'lon': -0.4543,  'color': [0, 255, 128, 200]},     # Zelená
    {'name': 'Tokyo (HND)',    'lat': 35.5494, 'lon': 139.7798, 'color': [255, 165, 0, 200]},     # Oranžová
    {'name': 'Dubai (DXB)',    'lat': 25.2532, 'lon': 55.3657,  'color': [138, 43, 226, 200]},    # Fialová
    {'name': 'Sydney (SYD)',   'lat': -33.9399, 'lon': 151.1753, 'color': [0, 191, 255, 200]},    # Modrá
    {'name': 'Cape Town (CPT)','lat': -33.9715, 'lon': 18.6021,  'color': [255, 215, 0, 200]}     # Žlutá
]

# 2. Vytvoříme všechny vzájemné kombinace tras mezi týmito letišti (pár z A do B)
routes = []
for orig in airports_list:
    for dest in airports_list:
        if orig['name'] != dest['name']:
            routes.append({
                'from_name': orig['name'],
                'to_name': dest['name'],
                'start_lon': orig['lon'],
                'start_lat': orig['lat'],
                'end_lon': dest['lon'],
                'end_lat': dest['lat'],
                'color': orig['color']
            })

df_clean_routes = pd.DataFrame(routes)

# 3. Pohled na celou planetu
view_state_network = pdk.ViewState(
    latitude=20.0,
    longitude=10.0,
    zoom=1.3,
    pitch=45,
    bearing=0
)

# 4. Definice ArcLayer pro přehlednou síť
arc_layer_clean = pdk.Layer(
    "ArcLayer",
    data=df_clean_routes,
    get_source_position=["start_lon", "start_lat"],
    get_target_position=["end_lon", "end_lat"],
    get_source_color="color",
    get_target_color="[255, 255, 255, 100]",  # Příletová strana přechází do bílé
    get_width=2.5,
    pickable=True
)

# 5. Vykreslení mapy
r4 = pdk.Deck(
    layers=[arc_layer_clean],
    initial_view_state=view_state_network,
    tooltip={"text": "Z letiště: {from_name}\nNa letiště: {to_name}"}
)

r4.to_html("ukol4_cista_sit.html", notebook_display=True)